# Hero Forge — research notebook

Generate an **Active-Theory-grade 3D website hero** from a single brief, in Colab.

The pipeline has **two layers**, and this notebook runs both:

| Layer | Owner | Produces |
|---|---|---|
| **1 — direction** | **Claude** (`colab/claude_layer`) | the `GenerationPlan` (scene, sky, lighting, camera), the **GLSL sky shader**, the standalone **Three.js renderer**, and the final assembly |
| **2 — assets** | **open-source models** (`services/generator`) | the actual pixels & geometry: SDXL concepts → TripoSR/InstantMesh `main.glb`, SDXL → Depth-Anything terrain `background.glb` |

Layer 1 is baked in (Claude already authored it). Layer 2 needs a **GPU runtime**
(Runtime → Change runtime type → GPU). Output is a self-contained `hero_site/`
you can open anywhere.

> **No fakes.** If a model can't run, its asset slot is reported `failed` and the
> AI-designed sky/light/camera still render — no placeholder geometry is invented.

## 0 · Get the code

In [ ]:
# Clone the repo (or skip if you've already uploaded the folder).
REPO = "https://github.com/gkjuwon-tech/3dwebsitegenerator"
BRANCH = "claude/ai-3d-interactive-generator-txt66v"
import os
if not os.path.isdir("3dwebsitegenerator"):
    !git clone --depth 1 -b $BRANCH $REPO
%cd 3dwebsitegenerator
print("repo root:", os.getcwd())

## 1 · Install

Core is light; the model stack is heavy and wants a GPU.

> **⚠️ Pillow pin + restart.** Colab ships Pillow 12, which removed
> `PIL._typing._Ink` and breaks `diffusers`' import. This cell pins Pillow back
> to `<12` **last**, so the resolved env is good. After it finishes you **must**
> do **Runtime → Restart session** once, then continue from the *next* cell
> (don't re-run this one).

In [ ]:
# Layer-1 core (always)
!pip -q install pydantic anthropic
# Layer-2 model stack (GPU). Comment out if you only want the environment.
!pip -q install diffusers transformers accelerate safetensors trimesh rembg
# TripoSR (preview reconstruction) ships from its repo, not PyPI:
!pip -q install git+https://github.com/VAST-AI-Research/TripoSR.git
# Pillow 12 removed PIL._typing._Ink → diffusers import fails. Pin back, LAST.
!pip -q install "pillow<12"
import PIL; print("pillow", PIL.__version__)
print("\n✅ Installed. NOW: Runtime → Restart session, then run the NEXT cell onward (skip this one).")

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("colab"))
sys.path.insert(0, os.path.abspath("services/generator"))
from claude_layer import load_baked, compile_live, list_baked, assemble_site
print("baked plans:", list_baked())

## 2 · Layer 1 — Claude authors the scene

Either load a plan Claude already authored (default, no API key), or compile a
fresh brief live (needs `ANTHROPIC_API_KEY`). The plan decides the sky palette,
sun direction, light rig, camera keyframes and the T2I prompts — per brief, not
from a template.

In [ ]:
USE_LIVE = False          # True → compile a fresh brief with Claude
BRIEF = "A floating obsidian monolith over a misty volcanic island at blue hour, slow heavy reveal, premium and ominous"

if USE_LIVE:
    os.environ.setdefault("ANTHROPIC_API_KEY", "")   # paste your key or set it in the env
    plan = compile_live(BRIEF)
else:
    plan = load_baked("obsidian_monolith")

print("project:", plan["scene"]["project_id"])
print("sky:", plan["sky"]["timeOfDay"], "| aurora:", plan["sky"]["aurora"]["enabled"],
      "| clouds:", plan["sky"]["clouds"]["enabled"])
print("lights:", [l["name"] for l in plan["lighting"]["lights"]])
print("notes:", plan["notes"][:200], "...")

## 3 · Layer 2 — open-source models render the assets

SDXL paints the concept images; TripoSR reconstructs the hero mesh; Depth-Anything
turns a terrain concept into a heightfield. Each stage is wrapped so a failure is
reported honestly rather than faked.

In [ ]:
from app.schema import ImageDirective, BackgroundSpec
from app import t2i, i23d, terrain

OUT = os.path.abspath("_work"); os.makedirs(OUT, exist_ok=True)
main_glb = main_fail = bg_glb = bg_fail = None
main_tris = bg_tris = 0
prov = {"compiler_model": "claude (baked)" , "licenses": ["OpenRAIL++-M","Apache-2.0","MIT"], "seeds": {}}

# ── main hero: SDXL concept → TripoSR ──
try:
    d = ImageDirective.model_validate(plan["main_image"])
    img = t2i.generate_image(d); prov["t2i_model"] = img.model; prov["seeds"]["main"] = img.seed
    img.image.save(os.path.join(OUT, "main_concept.png"))
    mesh = i23d.reconstruct_preview(img.image, OUT)
    main_glb, main_tris = mesh.glb_path, mesh.triangles; prov["main_i23d"] = mesh.model
    print("main.glb:", main_glb, main_tris, "tris")
except Exception as e:
    main_fail = f"{type(e).__name__}: {e}"; print("main FAILED:", main_fail)

In [ ]:
# ── background terrain: SDXL concept → Depth Anything heightfield ──
try:
    d = ImageDirective.model_validate(plan["background_image"])
    img = t2i.generate_image(d)
    img.image.save(os.path.join(OUT, "bg_concept.png"))
    spec = BackgroundSpec.model_validate(plan["scene"]["background"])
    terr = terrain.build_terrain(img.image, spec, OUT,
                                 max_triangles=int(plan["scene"]["constraints"]["max_background_triangles"]))
    bg_glb, bg_tris = terr.glb_path, terr.triangles; prov["background_pipeline"] = terr.model
    print("background.glb:", bg_glb, bg_tris, "tris")
except Exception as e:
    bg_fail = f"{type(e).__name__}: {e}"; print("background FAILED:", bg_fail)

## 4 · Assemble the hero site

Claude compiles the sky shader, writes the `ScenePackage`, drops in whatever
geometry the models produced, and emits a self-contained site.

In [ ]:
site = assemble_site(
    plan, os.path.abspath("hero_site"),
    main_glb=main_glb, main_triangles=main_tris, main_fail=main_fail,
    background_glb=bg_glb, background_triangles=bg_tris, background_fail=bg_fail,
    provenance=prov,
)
print("wrote", site)
import json
print(json.dumps(json.load(open(site + "/package.json"))["assets"], indent=2))

## 5 · Preview & download

In [ ]:
# Zip for download
import shutil
shutil.make_archive("hero_site", "zip", "hero_site")
try:
    from google.colab import files
    files.download("hero_site.zip")
except Exception:
    print("Not in Colab — grab hero_site.zip from the file browser.")

In [ ]:
# Live preview inside Colab (serves hero_site on a port and iframes it).
import http.server, socketserver, threading, functools, os
os.chdir("hero_site")
PORT = 8000
handler = functools.partial(http.server.SimpleHTTPRequestHandler)
threading.Thread(target=lambda: socketserver.TCPServer(("", PORT), handler).serve_forever(), daemon=True).start()
os.chdir("..")
try:
    from google.colab.output import serve_kernel_port_as_iframe
    serve_kernel_port_as_iframe(PORT, path="/index.html", height=560)
except Exception:
    print(f"Open http://localhost:{PORT}/index.html")

---
### Where to take it next
- Swap TripoSR for **InstantMesh** (`app.i23d.reconstruct_final`) for the final-quality hero — set `HERO_INSTANTMESH_DIR`.
- Turn on **live compile** to art-direct any brief on the fly.
- The `hero_site/` is portable: host the folder on any static CDN (R2/S3) and it runs standalone.